# BUILD ONCE — freeze the MISPR environment (no Docker)

**You run this notebook one time.** It builds the exact Python 3.8 environment the
workshop needs (MISPR, MDPropTools, FireWorks, MongoDB) and freezes it into a single
file, `mispr-env.tar.gz`, that students' Colab sessions download in ~2 minutes.

### How to use
1. **Runtime -> Run all.**
2. **Stay on this tab and stay active** ~25 minutes (Colab disconnects if idle).
3. At the end you get `mispr-env.tar.gz` saved to your Google Drive.
4. Download it from Drive to your laptop, then upload it to your GitHub **Release**.

You do **not** edit anything in this notebook. Just run it.


In [ ]:
#@title Step 1/6 - system build tools (~1 min)
import subprocess
def sh(c):
    print("$", c); subprocess.run(c, shell=True, check=True)
sh("apt-get -qq update")
sh("apt-get -qq install -y build-essential gfortran git wget bzip2 > /dev/null")
print("OK")


In [ ]:
#@title Step 2/6 - install Miniconda (Python 3.8) at /opt/conda (~1 min)
import os, subprocess
def sh(c):
    print("$", c); subprocess.run(c, shell=True, check=True)
if not os.path.exists("/opt/conda/bin/python"):
    sh("wget -q https://repo.anaconda.com/miniconda/Miniconda3-py38_23.11.0-2-Linux-x86_64.sh -O /tmp/mc.sh")
    sh("bash /tmp/mc.sh -b -p /opt/conda")
print(subprocess.run(["/opt/conda/bin/python","--version"],capture_output=True,text=True).stdout.strip())


In [ ]:
#@title Step 3/6 - conda packages incl. MongoDB (~4 min)
import subprocess
def sh(c):
    print("$", c); subprocess.run(c, shell=True, check=True)
sh("/opt/conda/bin/conda install -y -c conda-forge "
   "python=3.8 jupyter=1.0.0 jupyterlab=3.4.4 openbabel=3.1.1 conda-pack mongodb=5.0.3")
sh("/opt/conda/bin/conda clean -ya")
print("mongod at:", subprocess.run("ls /opt/conda/bin/mongod",shell=True,capture_output=True,text=True).stdout.strip())


In [ ]:
#@title Step 4/6 - pip packages (exact capsule pins) (~2 min)
import subprocess
def sh(c):
    print("$", c); subprocess.run(c, shell=True, check=True)
sh("/opt/conda/bin/pip install -U --no-cache-dir "
   "doepy==0.0.1 fireworks==1.9.7 llvmlite==0.36.0 numba==0.53.1 numpy==1.23.1 "
   "pandas==1.3.4 pymongo==3.12.0 scipy==1.7.1 seaborn==0.11.2 statsmodels==0.13.0")
sh("/opt/conda/bin/pip install -U --no-cache-dir --ignore-installed ruamel.yaml")
print("OK")


In [ ]:
#@title Step 5/6 - MolMD packages: pymatgen fork, custodian, ParmEd, mdproptools, mispr (~8 min)
import os, subprocess
def sh(c):
    print("$", c); subprocess.run(c, shell=True, check=True)
os.makedirs("/opt/src", exist_ok=True)
PIP = "/opt/conda/bin/pip"
def clone_install(url, name, branch=None, depth1=True):
    d = "/opt/src/" + name
    if not os.path.exists(d):
        depth = "--depth 1 " if (depth1 and not branch) else ""
        sh(f"git clone {depth}{url} {d}")
        if branch:
            sh(f"cd {d} && git checkout {branch}")
    sh(f"cd {d} && {PIP} install -e .")
clone_install("https://github.com/molmd/pymatgen.git",   "pymatgen", branch="molmd_fix", depth1=False)
clone_install("https://github.com/molmd/custodian.git",  "custodian")
clone_install("https://github.com/ParmEd/ParmEd.git",    "ParmEd")
clone_install("https://github.com/molmd/mdproptools.git","mdproptools")
clone_install("https://github.com/molmd/mispr.git",      "mispr")
print("OK")


In [ ]:
#@title Step 6/6 - smoke test, then freeze to mispr-env.tar.gz (~3 min)
import subprocess
def sh(c):
    print("$", c); subprocess.run(c, shell=True, check=True)
test = ("import pymatgen, custodian, parmed, mdproptools, mispr, fireworks, numba, seaborn; "
        "from mispr.gaussian.workflows.base.esp import get_esp_charges; "
        "from mispr.gaussian.utilities.fw_utilities import run_fake_gaussian; "
        "from mispr.lammps.utilities.utilities import run_fake_lammps; "
        "from mispr.lammps.workflows.base import lammps_workflow; "
        "print('IMPORTS OK')")
sh(f'/opt/conda/bin/python -c "{test}"')
sh("/opt/conda/bin/conda-pack -p /opt/conda -o /tmp/env.tar.gz --ignore-editable-packages --force")
sh("rm -rf /tmp/stage && mkdir -p /tmp/stage/opt/conda /tmp/stage/opt/src")
sh("tar -xzf /tmp/env.tar.gz -C /tmp/stage/opt/conda")
sh("cp -r /opt/src/. /tmp/stage/opt/src/")
sh("cd /tmp/stage && tar -czf /content/mispr-env.tar.gz opt")
sh("ls -lh /content/mispr-env.tar.gz")
print("\\nBUILT: /content/mispr-env.tar.gz")


In [ ]:
#@title Save the file to your Google Drive
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
os.makedirs('/content/drive/MyDrive/mispr_build', exist_ok=True)
shutil.copy('/content/mispr-env.tar.gz', '/content/drive/MyDrive/mispr_build/mispr-env.tar.gz')
print("Saved to Google Drive -> MyDrive/mispr_build/mispr-env.tar.gz")
print("Next: download it from Drive to your laptop, then upload to your GitHub Release.")
